In [ ]:
og_work_done = pd.read_csv('../data/work_done.csv')
work_commenced = pd.read_csv('../data/work_commenced.csv')
# initialise the index to the time column for time series analysis
og_work_done.index = pd.to_datetime(og_work_done.pop('Quarter'))
work_commenced.index = pd.to_datetime(work_commenced.pop('Quarter'))
og_work_done

In [ ]:
work_done = og_work_done.copy()
work_commenced

In [ ]:
print('====== INFORMATION ABOUT THE DATASETS ======')
print('== DATA TYPES ==')
print(work_done.dtypes)
print('== DATASET INDEX ==')
print(work_done.index)

In [ ]:
original_columns_mask_done = ['SA' not in col for col in work_done.columns]
work_done = work_done.loc[:, original_columns_mask_done]
work_done

In [ ]:
similar_times = list(set(work_done.index).intersection(set(work_commenced.index)))
similar_times.sort()
work_done = work_done.loc[similar_times]
work_commenced = work_commenced.loc[similar_times]
work_done

In [ ]:
work_commenced

In [ ]:
fig, axes = plt.subplots(figsize=(16, 9))
sns.lineplot(data=work_done, x=work_done.index, y=work_done.iloc[:, 0], ax=axes, label='Work Done', marker='o')
sns.lineplot(data=work_commenced, x=work_commenced.index, y=work_commenced.iloc[:, 0], ax=axes, label='Work Commenced')
axes.set_xlabel('Quarter (Q)')
axes.set_ylabel('Value of work done ($)')
axes.set_title('Value of Work Done vs Work Commenced in NSW since 1980')
plt.show()

## Observations
1. There is a noticeable upward trend in both the value of work done and work commenced.
2. More importantly, as of recently, the value of work commenced seems to often climb much higher than the value of work done, hinting at insufficiencies in the housing supply of NSW.
3. However, the data shows a lot of oscillatory behaviour due to the typical seasonal patterns with the housing industry such as supply being greater at certain times and lower at other times.

#### Improvements to make:
1. We must deseasonalize/adjust for seasonality for the Value of Work Commenced time series data to see the underlying trends more clearly.
2. Calculate a rolling average of both series to obtain curves that account for fluctuations and noise.

Value of Work Done already has a deseasonalized version provided by the ABS.

In [ ]:
# get the data of work done adjusted for seasonality by the ABS
work_done_AS = og_work_done.copy()
work_done_AS = work_done_AS.loc[:, ['SA' in col for col in work_done_AS.columns]]
vwd_deseasonalized = work_done_AS.iloc[:, 0]
vwd_deseasonalized.name = 'VWD Deseasonalized (NSW)'
vwd_deseasonalized

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
# deseasonalize the value of work commenced data. our data goes quarterly in terms of frequency
TIME_PERIOD = 4
vwc_data = work_commenced.iloc[:, 0]
decompose_vwc = seasonal_decompose(vwc_data, period=TIME_PERIOD, model='additive')

vwc_deseasonalized = vwc_data - decompose_vwc.seasonal
vwc_deseasonalized.name = 'VWC Deseasonalized (NSW)'
vwc_deseasonalized

In [ ]:
vwc_trend = vwc_deseasonalized.rolling(window=4, center=True).mean()
vwd_trend = vwd_deseasonalized.rolling(window=4, center=True).mean()
fig, axes = plt.subplots(figsize=(16, 9))
sns.lineplot(x=vwd_trend.index, y=vwd_trend, ax=axes, label='Work Done', marker='o')
sns.lineplot(x=vwc_trend.index, y=vwc_trend, ax=axes, label='Work Commenced', marker='o')
axes.set_xlabel('Quarter (Q)')
axes.set_ylabel('Monetary Value ($)')
axes.set_title('Value of Work Done vs Work Commenced in NSW since 1980 (Adjusting for seasonality, with rolling averages)')
plt.show()

## Observations
1. The VWD curve is lagging behind the VWC curve which makes sense because in theory, we expect there to be a gap in time between residential construction commencing and having the construction finish.
2. Overall, the value of housing has increased since 1980 with the steepest increase coming from the 2012 to 2017, where Australia experienced a housing and apartment construction boom due to factors like low interest rates and strong population growth.

**Correlation analysis with median house prices will require analysing the lagged VWD and VWC curves against the median house price curves.**